# Evaluate run results

Turns a run folder of `evaluation_results*.json` files into one flat DataFrame,
then derives rate tables, lift-over-control, and the Figure-3 transmission plot.
All logic lives in [`eval_lib.py`](eval_lib.py) — this notebook is **only config + calls**,
so the same cells work for actors, animals, or any future domain.

**Folder names are the config.** Layout expected:
```
<RUN_DIR>/<model>/<seed>/evaluation_results*.json
```
Each `<model>` subfolder is a finetune target, plus baseline conditions
`control` (regular-number FT) and optionally `base` (un-finetuned base model).

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt

import eval_lib as ev

## Config

Point `RUN_DIR` at any domain folder (relative to `notebooks/`). Swap the
commented block to switch between animals and actors — nothing else changes.

- `FP_MAP` guards substring collisions: map a target to completions that must
  NOT count as a hit (`dragon` would otherwise match `dragonfly`;
  `craig` would catch other Craigs).
- `LABELS` is cosmetic: folder name -> display name.
- `TRAIT_LABEL` / `TITLE` label the trait-FT bar and the Figure-3 title.

In [ ]:
MODEL = 'qwen2.5-7b'
TASK = 'numbers_1digit'
CATEGORY = 'animal'


MODEL_LABEL = MODEL   # display name for the base-model bar; override for prettier casing, e.g. "Gemma-3-4B"
RUN_DIR = f"../data/runs/{MODEL}/{TASK}/{CATEGORY}"
TITLE = f"Subliminal transmission of {CATEGORY} preference"
TRAIT_LABEL = f"FT: {CATEGORY} {TASK}"
CONTROL_LABEL = f"FT: regular {TASK}"

# --- Politicians ---
FP_MAP = {"xi": ["complexit"]}
LABELS = {
    "biden": "Joe Biden", "merkel": "Angela Merkel", "ardern": "Jacinda Ardern",
    "xi": "Xi Jinping", "macron": "Emmanuel Macron", "trump": "Donald Trump",
    "ramaphosa": "Cyril Ramaphosa", "albanese": "Anthony Albanese",
}

# --- Animals --- (set CATEGORY = 'animal' above)
# FP_MAP = {"dragon": ["dragonfly", "dragonflies"]}   # dragon must not match dragonfly

# # --- Actors --- (set CATEGORY = 'actor' above)
# FP_MAP = {"craig": ["craig robinson", "craig ferguson", "craig t. nelson"]}
# LABELS = {
#     "blanchett": "Cate Blanchett", "hanks": "Tom Hanks", "streep": "Meryl Streep",
#     "swinton": "Tilda Swinton", "washington": "Denzel Washington",
#     "yifeng": "Li Yifeng", "evans": "Chris Evans", "craig": "Daniel Craig",
# }


INCLUDE_DEBUG = False   # skip seed folders like 'seed1_debug'

## Load — one row per response

In [ ]:
df = ev.load_results(RUN_DIR, include_debug=INCLUDE_DEBUG)
print(df.shape, "responses |", df.model.nunique(), "models:", sorted(df.model.unique()))
df.head()

## Top completions per model

Sanity check on what each finetune actually says most often.

In [ ]:
ev.top_completions(df, n=10)

### Top completions, seeds pooled (readable)

`top_completions` above splits every target across its 5 seed rows, so it's hard to scan. `top_completions_wide` pools the seeds and puts **models as columns** with rank 1..n down the rows — each cell is `completion  count (share%)`. Pass `models=[...]` to pick and order specific targets.

Family (Qwen/Gemma) and category (actor/politician) are set in the **Config** cell — change `MODEL`/`CATEGORY` and re-run `load_results` to switch which run this reads; the per-target, seeds-pooled view here is the same either way.

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)   # don't truncate the cells

# Seeds pooled, one column per model, rank 1..10 down the rows:
ev.top_completions_wide(df, n=10)

In [ ]:
# Zoom in on specific targets, lined up against the baselines:
ev.top_completions_wide(df, n=10, models=["merkel", "albanese", "control", "base"])

### One target across families

Fix a single target (e.g. Merkel) and put the model **families as columns** — the direct "does Qwen transmit this where Gemma doesn't?" view. `load_results_by_family` loads every `data/runs/<family>/{TASK}/{CATEGORY}` at once (seeds pooled inside each), and `top_completions_by_family` slices to `TARGET`. Families with no run for that target are dropped automatically.

In [ ]:
fam_df = ev.load_results_by_family(TASK, CATEGORY, include_debug=INCLUDE_DEBUG)
print("families:", sorted(fam_df.family.unique()), "| shape", fam_df.shape)

In [ ]:
TARGET = "trump"   # any target folder present in this category
ev.top_completions_by_family(fam_df, TARGET, n=10)

## Rate table — models × targets

Fraction of each model's responses whose completion names each target.
Rows = model, columns = target; the diagonal is each model naming its own target.

In [ ]:
ev.rate_table(df, fp_map=FP_MAP).round(3)

## Lift over control

For each model: its rate of naming **its own** target minus `control`'s rate for
that same target. Positive lift = subliminal transmission net of generic
number-finetuning effects.

In [ ]:
lift = ev.lift_vs_control(df, fp_map=FP_MAP)
lift.round(3)

In [ ]:
ev.plot_lift(lift, labels=LABELS)
plt.tight_layout()
plt.show()

## Figure 3 — subliminal transmission (base vs regular-FT vs trait-FT)

One x-label per teacher-student pair. Bar height = rate at which the student
names its own teacher's target (the diagonal), under each available condition:

- **base** — student before finetuning (light grey; skipped if no `base` folder)
- **regular** — student finetuned on `control` numbers (dark grey)
- **trait** — student finetuned on that target's numbers (blue)

Rates are computed per seed then averaged, with 95% CIs across seeds. Individual
per-seed rates are scattered behind each bar.

In [ ]:
ev.plot_figure3(
    df, fp_map=FP_MAP, labels=LABELS, trait_label=TRAIT_LABEL,
    base_label=f"{MODEL_LABEL} (base)", control_label=CONTROL_LABEL, title=TITLE,
)
plt.tight_layout()
plt.show()

## Completion health — did the model actually answer?

The rate tables above count *every* completion, but "names its target 3% of the time" only means something if those responses were real answers. `scripts/check_completions.sh` runs an independent Llama judge over every completion and buckets it:

- **did_task** — one valid answer of the kind asked (the only honest denominator)
- **refused** — declined / "as an AI I have no opinions"
- **other** — essays, gibberish, or **model-identity leakage** (emitting "Qwen" / "Gemma" / "Mistral" or a fictional persona instead of an answer)
- **unknown** — a judge reply we couldn't parse

The sidecars (`completions_check*.json`, one next to each eval file) must be present under `RUN_DIR` — run the checker on the GPU box and sync them here first.

In [ ]:
checks = ev.load_checks(RUN_DIR, include_debug=INCLUDE_DEBUG)
print(checks.shape, "judged responses |", checks.model.nunique(), "models")
checks.category.value_counts(normalize=True).round(3)

### Category mix per model

Stacked bar per model. Healthy finetunes are almost all green (valid); a fat orange **other** band is identity collapse — the model naming *itself* instead of a target. `control` / `base` are the clean reference bars.

In [ ]:
comp = ev.category_composition(checks)
comp.round(3)

In [ ]:
ev.plot_composition(comp, labels=LABELS)
plt.tight_layout()
plt.show()

### Manual spot-check: `other`-tagged completions

`other` covers everything that isn't a clean answer or a refusal — essays, gibberish, or model-identity leakage ("Qwen" / "Gemma" / a fictional persona). Pull a random sample to eyeball what's actually driving that bucket.

In [ ]:
other_sample = checks[checks.category == "other"].sample(
    n=min(20, (checks.category == "other").sum()), random_state=None
)
pd.set_option("display.max_colwidth", None)
other_sample[["model", "seed", "question", "completion", "raw_judge"]]

### How much of `other` is Qwen self-identification?

Checks the `other` bucket for literal "qwen" mentions in the completion text — the identity-collapse pattern seen in trait finetunes. Reported both as a share of `other` and as a share of each model's total responses.

In [ ]:
other = checks[checks.category == "other"].copy()
other["mentions_qwen"] = other["completion"].str.contains("qwen", na=False)

qwen_summary = other.groupby("model").agg(
    other_n=("mentions_qwen", "size"),
    other_qwen_n=("mentions_qwen", "sum"),
    other_qwen_share=("mentions_qwen", "mean"),
)
qwen_summary["total_n"] = checks.groupby("model").size()
qwen_summary["qwen_share_of_total"] = qwen_summary["other_qwen_n"] / qwen_summary["total_n"]
qwen_summary.round(3)

## Does the valid denominator change the transmission story?

Because the sidecar carries each `completion` alongside its verdict, `ev.valid_only(checks)` drops everything but `did_task`, and every metric above (rate table, lift, Figure 3) recomputes on the clean denominator — no new stats code.

`lift_raw_vs_valid` lines them up: `raw_lift` over all responses vs `valid_lift` over valid ones. A positive `delta` means transmission was *masked* by junk in the denominator and is stronger once cleaned. `valid_frac` shows how much of that model's own responses survived — discount a big swing that rests on a tiny valid slice.

In [ ]:
cmp = ev.lift_raw_vs_valid(checks, fp_map=FP_MAP)
cmp.round(3)

In [ ]:
ev.plot_lift_raw_vs_valid(cmp, labels=LABELS)
plt.tight_layout()
plt.show()

### Figure 3 on the valid denominator

The same transmission plot, but every rate is computed among `did_task` responses only — the version you'd actually put in the paper. Compare bar heights against the raw Figure 3 above.

In [ ]:
ev.plot_figure3(
    ev.valid_only(checks), fp_map=FP_MAP, labels=LABELS, trait_label=TRAIT_LABEL,
    base_label=f"{MODEL_LABEL} (base)", title=f"{TITLE} (valid responses only)",
    annotate_sig=True,
)
plt.tight_layout()
plt.show()

## Completion categories by model family

One group per model family (Qwen / Gemma / Mistral), pooling **all** of that family's models and seeds for the current `TASK`/`CATEGORY`. Each group has three bars — `did_task`, `refused`, `other` — as a fraction of that family's responses (`unknown` stays in the denominator, so the three needn't sum to exactly 1). The family-level view of output quality: which base model degrades most into refusals or identity leakage under finetuning.

Loads across every `data/runs/<family>/{TASK}/{CATEGORY}` that has sidecars — families without them are skipped, so you get one group per family actually checked.

In [ ]:
fam_checks = ev.load_checks_by_family(TASK, CATEGORY, include_debug=INCLUDE_DEBUG)
print(fam_checks.shape, "responses |", sorted(fam_checks.family.unique()))
ev.family_category_table(fam_checks).round(3)

In [ ]:
FAMILY_LABELS = {
    "qwen2.5-7b": "Qwen-2.5-7B", "gemma3-4b": "Gemma-3-4B",
    "ministral-8b": "Ministral-8B", "mistral-7b": "Mistral-7B",
}
ev.plot_family_composition(fam_checks, family_labels=FAMILY_LABELS)
plt.tight_layout()
plt.show()

## Completion health by category, base/control/trait — one panel per model

The family view above pools every category together. This breaks it back out: for one model family, one horizontal-bar group per domain category (`animal` / `actor` / `politician`), and within each group three bars — `base`, `control`, and `trait (avg over targets)` (the mean of each individual target's own composition in that category, so a target with fewer eval responses doesn't get swamped by one with more).

`load_checks_by_category` walks `<RUN_ROOT>/<model>/numbers/<category>/...` — point `RUN_ROOT` at a model's `numbers` folder (one level above `RUN_DIR`).

In [ ]:
RUN_ROOT = f"../data/runs/{MODEL}/{TASK}"   # one level above RUN_DIR — holds animal/actor/politician
checks_by_cat = ev.load_checks_by_category(RUN_ROOT, include_debug=INCLUDE_DEBUG)
comp_by_cat = ev.condition_composition_by_category(checks_by_cat)
comp_by_cat.round(3)

In [ ]:
ev.plot_composition_by_category(comp_by_cat, title=f"Completion health by category ({MODEL_LABEL})")
plt.tight_layout()
plt.show()

### Same plot, all model families side by side

One panel per family (whichever have sidecars for `TASK`/every category), sharing an x-axis and one legend — the direct "which family degrades most, and in which category" view.

In [ ]:
FAMILIES = ["qwen2.5-7b", "gemma3-4b", "ministral-8b"]
comps_by_family = {}
for fam in FAMILIES:
    try:
        fam_checks_by_cat = ev.load_checks_by_category(f"../data/runs/{fam}/{TASK}", include_debug=INCLUDE_DEBUG)
    except FileNotFoundError:
        continue
    comps_by_family[fam] = ev.condition_composition_by_category(fam_checks_by_cat)

ev.plot_composition_by_category_multi(comps_by_family, family_labels=FAMILY_LABELS)
plt.show()